# 강의 04 · 실습 5 — 기성 MCP 서버 연결 · (1) 강사 시연

## 1. 문제상황

- 해외 지사가 여러 곳인 회사의 회의 일정 담당자는 회의 시간을 잡을 때마다 지사별 현재 시각과 시차를 확인합니다.
- 지금 서울이 몇 시인지, 서울 오후 3시가 뉴욕에서는 몇 시인지를 담당자가 매번 시간대 표를 보고 계산합니다.
- 현재 시각과 시간대 변환을 알려 주는 프로그램은 이미 다른 사람이 만들어 공개해 두었습니다.
- 그런데 그 프로그램을 우리 에이전트에 붙이려면 호출 방법을 따로 알아내 연결 코드를 새로 짜야 합니다.

## 2. 문제와 목표

- **문제**: 현재 시각 조회와 시간대 변환을 사람이 손으로 계산합니다. 이미 공개된 프로그램이 있는데도, 에이전트에 붙이려면 연결 코드를 새로 짜야 합니다.
- **목표**: 공개된 시간 서버를 MCP(Model Context Protocol) 규격으로 에이전트에 연결해, 모델이 질문에 따라 도구를 골라 호출하는 실행 흐름을 만듭니다. 서버 코드는 한 줄도 짜지 않고, 연결 선언만 적습니다.
    - 시간 서버: `uvx mcp-server-time`으로 띄우는 공개 MCP 서버입니다. 현재 시각을 알려 주는 도구와 시간대를 변환하는 도구 두 개를 내놓습니다.
    - 연결 선언: 서버 이름·실행 명령·인자·전송 방식(`stdio`)을 적은 딕셔너리 하나입니다.
    - 실행 흐름: 도구를 묶은 모델을 부르는 model 노드와 도구를 실행하는 tools 노드를 조건부 엣지로 이은 그래프입니다. 질문 두 개(현재 시각·시간대 변환)는 코드에 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 첫 질문에서는 현재 시각 도구를, 둘째 질문에서는 시간대 변환 도구를 모델이 골라 호출합니다.
    - 도구가 돌려준 값을 바탕으로 최종 답을 쓰는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex05_s1_diagram.svg)

## 4. 단계별 요구사항

1. **서버 연결을 선언합니다.**
    - `MultiServerMCPClient`에 서버 이름 `time`으로, 실행 명령 `uvx`, 인자 `mcp-server-time`과 `--local-timezone=Asia/Seoul`, 전송 방식 `stdio`를 적습니다.
    - 서버를 어떻게 띄울지만 적고, 서버 코드는 짜지 않습니다.
2. **도구 목록을 받아 옵니다.**
    - `await client.get_tools()`로 서버가 내놓는 도구 목록을 받아, 도구마다 이름과 설명의 첫 줄을 출력합니다.
3. **상태를 정의하고 도구를 모델에 묶습니다.**
    - 대화 기록(`messages`) 키 하나를 가지는 상태를 `add_messages` 리듀서와 함께 선언하고, 받아 온 도구 목록을 `llm.bind_tools(tools)`로 모델에 묶습니다.
4. **실행 그래프를 연결합니다.**
    - model 노드는 도구를 묶은 모델을 호출해 응답 메시지를 `messages` 키에 쌓습니다.
    - tools 노드는 `ToolNode(tools)`로 만듭니다.
    - 판단 함수는 마지막 메시지에 `tool_calls`가 있으면 `"tools"`를, 없으면 `END`를 돌려줍니다.
    - START→model 고정 엣지, model 뒤 조건부 엣지, tools→model 고정 엣지를 추가하고 컴파일합니다.
5. **그래프를 실행합니다.**
    - 현재 시각을 묻는 질문과 시간대 변환을 묻는 질문을 차례로 넣고, 메시지마다 종류와 내용(도구 호출이면 도구 이름과 인자)을 출력합니다.
    - 그래프는 비동기로 실행하므로 `await app.ainvoke(…)`를 씁니다.

## 5. 코드 골격 — MCP 클라이언트 4단

MCP 서버를 에이전트에 붙이는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다. ④ 실행 그래프 연결은 상태 정의·노드 함수·노드 등록·엣지 연결·컴파일과 실행의 다섯 단계를 한 단으로 묶은 것입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 서버 연결 선언 | 어느 서버를 어떤 방식으로 띄울지 딕셔너리에 적습니다 | `MultiServerMCPClient({…, "transport": "stdio"})` | 1 |
| ② 도구 목록 수령 | 서버가 가진 도구 목록을 받아 옵니다 | `tools = await client.get_tools()` | 2 |
| ③ 모델에 묶기 | 받아 온 도구를 그대로 모델에 붙입니다 | `bound = llm.bind_tools(tools)` | 3 |
| ④ 실행 그래프 연결 | 모델 노드와 도구 노드를 등록하고 조건부 엣지로 연결해 컴파일하고 실행합니다 | `StateGraph(State)`, `ToolNode(tools)`, `compile()`, `await app.ainvoke(…)` | 4, 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- MCP 서버는 `uvx`가 띄웁니다. `uvx`는 `uv`를 설치할 때 함께 설치되는 실행 파일이며, 파이썬 패키지를 받아 그 자리에서 실행합니다. 처음 실행할 때는 패키지를 내려받느라 시간이 조금 걸립니다.
- 노트북(Jupyter) 안에서는 표준 오류(`sys.stderr`)가 화면 출력용 객체로 바뀌어 있어, 서버 프로세스를 띄울 때 실제 파일이 필요한 곳에서 오류가 납니다. 그래서 MCP 클라이언트를 불러오기 전에 `sys.stderr = sys.__stderr__`로 원래 표준 오류로 되돌립니다. 파이썬 스크립트로 실행할 때는 이 줄이 필요 없습니다.

In [5]:
import os
import sys

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

sys.stderr = sys.__stderr__   # 노트북 안에서 MCP 서버 프로세스를 띄우려면 실제 표준 오류 파일이 필요합니다 (이 줄은 MCP 클라이언트를 불러오기 전에 둡니다)
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")


def show_messages(msgs: list) -> None:
    """대화 기록을 메시지 종류와 내용(도구 호출이면 도구 이름과 인자)으로 한 줄씩 출력한다."""
    for m in msgs:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"    [{kind}] tool_calls={[(c['name'], c['args']) for c in calls]}")
        elif m.content:
            print(f"    [{kind}] {str(m.content)[:200]}")


print("모델 준비를 마쳤습니다.")

모델 준비를 마쳤습니다.


### 단계 ① — 서버 연결 선언 (요구사항 1)

`MultiServerMCPClient`에 넘기는 딕셔너리가 연결 선언의 전부입니다. 키 `time`은 우리가 붙인 서버 이름이고, `command`와 `args`가 서버를 띄우는 실행 명령이며, `transport`가 `"stdio"`이면 같은 컴퓨터에서 프로세스로 띄워 표준 입출력으로 주고받습니다. 이 셀은 선언만 하고 서버를 아직 띄우지 않습니다.

In [6]:
client = MultiServerMCPClient({
    "time": {
        "command": "uvx",
        "args": [
            "mcp-server-time",
            "--local-timezone=Asia/Seoul",
        ],
        "transport": "stdio",
    }
})

# print("선언한 서버: time (uvx mcp-server-time, stdio)")


선언한 서버: time (uvx mcp-server-time, stdio)


### 단계 ② — 도구 목록 수령 (요구사항 2)

`get_tools()`가 서버를 띄우고 서버가 내놓는 도구 목록을 받아 옵니다. 서버가 무엇을 내놓는지는 코드가 아니라 연결한 뒤에 확인합니다. 비동기 함수이므로 `await`를 붙입니다. 받아 온 목록의 원소는 그대로 모델에 묶을 수 있는 도구입니다.

In [7]:
tools = await client.get_tools()

print("서버가 준 도구:")
for t in tools:
    print(f"    {t.name} — {t.description.splitlines()[0]}")

서버가 준 도구:
    get_current_time — Get current time in a specific timezone
    convert_time — Convert time between timezones


### 단계 ③ — 상태 정의와 모델에 묶기 (요구사항 3)

상태는 `messages` 한 키입니다. `add_messages` 리듀서가 노드마다 돌려주는 메시지를 뒤에 쌓습니다. `bind_tools(tools)`는 모델에 도구 목록을 알려, 모델이 답 대신 도구 호출 요청(`tool_calls`)을 돌려줄 수 있게 합니다. 서버에서 받아 온 도구를 그대로 넘깁니다.

In [11]:
class State(TypedDict):
    messages: Annotated[list, add_messages]   # 질문·도구 호출 요청·도구 결과·답이 쌓이는 대화 기록


bound = llm.bind_tools(tools)
print("모델에 묶은 도구:", [t.name for t in tools])

모델에 묶은 도구: ['get_current_time', 'convert_time']


### 단계 ④ — 실행 그래프 연결과 실행 (요구사항 4, 5)

상태 정의 → 노드 함수 → 노드 등록 → 엣지 연결 → 컴파일과 실행의 순서를 따릅니다. 조건부 엣지의 판단 함수는 마지막 메시지에 `tool_calls`가 있는지만 봅니다. tools 노드는 직접 짜지 않고 `ToolNode(tools)`가 대신합니다. 아래 두 셀(④-a, ④-b)이 모두 이 한 단계에 속합니다.

#### 단계 ④-a — 노드·엣지 연결과 컴파일

In [12]:
def call_model(state: State) -> dict:
    """도구를 묶은 모델을 호출해 응답 메시지를 대화 기록에 쌓는다."""
    msg = bound.invoke(state["messages"])
    return {"messages": [msg]}


def should_continue(state: State) -> str:
    """마지막 메시지에 도구 호출 요청이 있으면 tools로, 없으면 END로 보낸다."""
    msgs = state["messages"]
    return "tools" if msgs[-1].tool_calls else END


g = StateGraph(State)
g.add_node("model", call_model)
g.add_node("tools", ToolNode(tools))

g.add_edge(START, "model")
g.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
g.add_edge("tools", "model")

app = g.compile()

print("등록한 노드:", list(g.nodes))

등록한 노드: ['model', 'tools']


#### 단계 ④-b — 실행

질문 두 개를 차례로 넣습니다. 그래프 실행도 비동기이므로 `await app.ainvoke(…)`를 씁니다. 메시지 종류마다 무엇이 출력되는지 봅니다.

In [10]:
QUESTIONS = [
    "지금 서울은 몇 시야? 도구로 확인해줘.",
    "서울 시각으로 오늘 오후 3시는 뉴욕에서 몇 시야? 도구로 변환해줘.",
]

for i, q in enumerate(QUESTIONS, 1):
    print(f"=== {i}번 질문: {q} ===")
    out = await app.ainvoke({"messages": [HumanMessage(q)]})
    show_messages(out["messages"])
    print()

=== 1번 질문: 지금 서울은 몇 시야? 도구로 확인해줘. ===
    [HumanMessage] 지금 서울은 몇 시야? 도구로 확인해줘.
    [AIMessage] tool_calls=[('get_current_time', {'timezone': 'Asia/Seoul'})]
    [ToolMessage] [{'type': 'text', 'text': '{\n  "timezone": "Asia/Seoul",\n  "datetime": "2026-09-15T10:44:42+09:00",\n  "day_of_week": "Tuesday",\n  "is_dst": false\n}', 'id': 'lc_cc647850-d8da-4eec-a6a2-e5444404da6
    [AIMessage] 현재 서울은 **2026년 9월 15일 화요일 오전 10시 44분**입니다.

=== 2번 질문: 서울 시각으로 오늘 오후 3시는 뉴욕에서 몇 시야? 도구로 변환해줘. ===
    [HumanMessage] 서울 시각으로 오늘 오후 3시는 뉴욕에서 몇 시야? 도구로 변환해줘.
    [AIMessage] tool_calls=[('convert_time', {'source_timezone': 'Asia/Seoul', 'time': '15:00', 'target_timezone': 'America/New_York'})]
    [ToolMessage] [{'type': 'text', 'text': '{\n  "source": {\n    "timezone": "Asia/Seoul",\n    "datetime": "2026-09-15T15:00:00+09:00",\n    "day_of_week": "Tuesday",\n    "is_dst": false\n  },\n  "target": {\n    "
    [AIMessage] 서울 시각 **오늘 오후 3시**는 뉴욕에서 **오늘 오전 2시**입니다. (뉴욕 서머타임 적용)



## 7. 실행 결과 확인

위 실행 결과에서 다음 네 가지를 확인합니다.

1. 단계 ②의 출력에 서버가 준 도구 두 개(`get_current_time`, `convert_time`)와 설명이 출력됩니다. 이 도구들은 우리 코드 어디에도 정의되어 있지 않습니다.
2. 첫 질문의 기록은 `HumanMessage` → `AIMessage`(tool_calls에 `get_current_time`) → `ToolMessage`(서버가 돌려준 시각) → `AIMessage`(최종 답) 순서입니다. 모델이 도구를 고르고, tools 노드가 실행하고, 모델이 그 결과로 답을 썼습니다.
3. 둘째 질문의 기록에서는 `tool_calls`의 도구 이름이 `convert_time`이고, 인자에 서울과 뉴욕의 시간대와 시각이 들어 있습니다. 질문이 달라지면 모델이 다른 도구를 고릅니다.
4. 두 질문 모두 마지막 메시지는 `tool_calls`가 없는 `AIMessage`입니다. 판단 함수가 `END`를 돌려주었기 때문입니다.